# Instructor Notebook 02 — Deep Learning
**ComplianceGPT Lab · REU 2026 · Week 2**

> **Teaching script:** Run every cell top-to-bottom. All random_state values fixed.

**Learning arc:**
1. Show what ML *can't* do (nonlinear boundaries)
2. Neural network = layers of learned transformations that solve nonlinear problems
3. Backpropagation — training loop intuition
4. Scale: depth + GPUs = modern AI
5. Apply to HIPAA — how MLP compares to hand features

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)

print('Libraries loaded.')

---
## Part 1 · The Wall — What Logistic Regression Can't Do

> **Say:** "We showed that Logistic Regression draws a straight line (or flat plane) to separate classes. But real patterns in the world aren't always linearly separable."

> **Say:** "The classic example is XOR. Watch what happens."

In [ ]:
# XOR — the simplest problem a linear model cannot solve
# XOR(A, B) = 1 only when exactly one of A, B is 1
X_xor = np.array([[0,0],[0,1],[1,0],[1,1]])
y_xor = np.array([0, 1, 1, 0])

print('XOR problem:')
print('A  B  |  XOR')
print('------+-----')
for (a, b), label in zip(X_xor, y_xor):
    print(f'{a}  {b}  |  {label}')
print()
print('A linear classifier must draw ONE straight line to separate 0s from 1s.')
print('Try drawing it. You cannot. The 1s are in opposite corners.')

In [ ]:
# Logistic Regression fails on XOR
lr = LogisticRegression()
lr.fit(X_xor, y_xor)
lr_preds = lr.predict(X_xor)

print('Logistic Regression on XOR:')
print(f'  Predictions: {lr_preds}')
print(f'  True labels: {y_xor}')
print(f'  Accuracy: {accuracy_score(y_xor, lr_preds):.0%}')
print()
print('  → Best it can do is 50%. A coin flip.')
print('  → No straight line can solve XOR.')

In [ ]:
# Neural Network solves XOR — with one hidden layer
mlp = MLPClassifier(hidden_layer_sizes=(4,), activation='relu', max_iter=10000, random_state=42)
mlp.fit(X_xor, y_xor)
mlp_preds = mlp.predict(X_xor)

print('Neural Network (1 hidden layer, 4 neurons) on XOR:')
print(f'  Predictions: {mlp_preds}')
print(f'  True labels: {y_xor}')
print(f'  Accuracy: {accuracy_score(y_xor, mlp_preds):.0%}')
print()
print('  → 100%. The hidden layer learned a new coordinate system where XOR is linearly separable.')
print()
print('Network architecture:')
print('  Input (2) → Hidden (4 neurons, ReLU) → Output (2 classes)')
print(f'  Total parameters: {sum(w.size for w in mlp.coefs_) + sum(b.size for b in mlp.intercepts_)}')

---
## Part 2 · What a Neural Network Actually Is

> **Say:** "A neural network is a function with learnable parameters. You compose many simple functions (neurons) into layers. Each layer transforms the data into a new representation. After many layers, the final representation is linearly separable."

**One neuron:**
```
output = ReLU(w₁·x₁ + w₂·x₂ + ... + wₙ·xₙ + bias)
```
- `w` = weights (learned during training)
- `bias` = offset (also learned)
- `ReLU(z) = max(0, z)` — the non-linearity that makes it powerful

In [ ]:
# Manual demonstration of one neuron
def relu(z):
    return max(0, z)

def single_neuron(x1, x2, w1=0.7, w2=0.3, bias=-0.2):
    """One neuron: weighted sum → ReLU"""
    z = w1 * x1 + w2 * x2 + bias
    return relu(z), z

print('Single neuron output (w1=0.7, w2=0.3, bias=-0.2):')
print()
for x1, x2 in [(0,0),(0,1),(1,0),(1,1)]:
    out, z = single_neuron(x1, x2)
    print(f'  Input ({x1},{x2})  z = {z:+.2f}  ReLU(z) = {out:.2f}')

print()
print('The network has millions of these neurons arranged in layers.')
print('Training finds the w and bias values that minimize prediction error.')

---
## Part 3 · Backpropagation — How It Learns

> **Say:** "Training a neural network is a loop. Forward pass: input → prediction. Loss: how wrong? Backward pass: how much did each weight contribute to the error? Update: nudge every weight to reduce error. Repeat millions of times."

> **Say:** "Let's watch this loop on real data. I'll train a network and watch the loss drop."

In [ ]:
# Simulate the training loop on XOR
# We'll manually track loss at each epoch
from sklearn.neural_network import MLPClassifier
import numpy as np

# Use warm_start to train one epoch at a time and track loss
loss_curve = []

net = MLPClassifier(
    hidden_layer_sizes=(8, 4),
    activation='relu',
    max_iter=1,
    warm_start=True,
    random_state=42,
    learning_rate_init=0.1
)

# Train 200 epochs, record loss each time
for epoch in range(200):
    net.fit(X_xor, y_xor)
    loss_curve.append(net.loss_)

# Display key milestones
print('Training loop — loss at key epochs:')
print()
milestones = [0, 9, 24, 49, 99, 149, 199]
for ep in milestones:
    bar_len = int((1 - min(loss_curve[ep], 1.0)) * 30)
    bar = '█' * bar_len + '░' * (30 - bar_len)
    print(f'  Epoch {ep+1:>3} | Loss: {loss_curve[ep]:.4f} | Progress: [{bar}]')

print()
print(f'Final accuracy: {accuracy_score(y_xor, net.predict(X_xor)):.0%}')

---
## Part 4 · Why Depth Matters

> **Say:** "Deeper networks can learn more complex functions. Each layer refines the representation. Early layers learn simple patterns (edges, word n-grams). Later layers learn abstract concepts (shapes, legal conditions)."

> **Say:** "The AlexNet breakthrough in 2012: a deep network on GPUs dropped ImageNet error from 26% to 15% overnight. The same logic applies to text. GPT started at 117M parameters. GPT-4 is estimated at ~1.8 trillion."

In [ ]:
# Generate a harder non-linear problem (checkerboard pattern)
np.random.seed(42)
n = 200
X_hard = np.random.uniform(-3, 3, (n, 2))
# Checkerboard: alternating squares = very nonlinear
y_hard = ((np.floor(X_hard[:, 0]) + np.floor(X_hard[:, 1])) % 2).astype(int)

scaler = StandardScaler()
X_hard_s = scaler.fit_transform(X_hard)

architectures = {
    'Logistic Regression (0 hidden)': LogisticRegression(),
    'Shallow MLP  (1 layer, 8)':      MLPClassifier(hidden_layer_sizes=(8,), max_iter=1000, random_state=42),
    'Deeper MLP   (2 layers, 32×16)': MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=1000, random_state=42),
    'Deep MLP     (4 layers, 64×32×16×8)': MLPClassifier(hidden_layer_sizes=(64, 32, 16, 8), max_iter=2000, random_state=42),
}

print('Checkerboard classification — harder than XOR, requires depth:')
print()
print(f"  {'Architecture':<40} {'5-fold CV Accuracy':>18}")
print('  ' + '-' * 62)
for name, model in architectures.items():
    scores = cross_val_score(model, X_hard_s, y_hard, cv=5, scoring='accuracy')
    bar = '█' * int(scores.mean() * 20)
    print(f"  {name:<40} {scores.mean():>17.1%}  {bar}")

print()
print('  → More layers → better at complex patterns')
print('  → But: more layers = more parameters = need more data and more compute')

---
## Part 5 · Scale — Parameters and GPUs

> **Say:** "What makes modern LLMs different from the MLP we just trained isn't the concept — it's the scale. Same math, 10 billion times more parameters, trained on 10 trillion tokens."

In [ ]:
scale_table = [
    ('Perceptron (1958)',        1,              'punch cards',    'XOR impossible'),
    ('Early MLP (1980s)',        1_000,          'CPU seconds',    'toy problems'),
    ('LeNet (1998)',             60_000,         'CPU hours',      'MNIST digits'),
    ('AlexNet (2012)',          61_000_000,      'GPU days',       'ImageNet photos'),
    ('BERT-base (2018)',       110_000_000,      'GPU weeks',      'NLP understanding'),
    ('GPT-3 (2020)',         175_000_000_000,    'GPU months',     'few-shot learning'),
    ('Gemma3-4B (2025)',      4_000_000_000,     '≈4 GPU-days',    'our research model'),
    ('GPT-4 (est., 2023)', 1_800_000_000_000,   'months (est.)',  'frontier model'),
]

print(f"  {'Model':<28} {'Parameters':>18}  {'Training':>14}  Notes")
print('  ' + '-' * 85)
for model, params, compute, notes in scale_table:
    print(f"  {model:<28} {params:>18,}  {compute:>14}  {notes}")

print()
print('  A parameter is one learned weight in the network.')
print('  Gemma3-4B has 4 billion of them. It fits on one V100 GPU on the cluster.')

---
## Part 6 · Back to HIPAA — Does Depth Help?

> **Say:** "Let's go back to the HIPAA data. We have hand-crafted boolean features (from the LLM extraction). Can we do better than Logistic Regression by using a deep network on those same features?"

In [ ]:
import json

HIPAA_PATH = '/Users/priscilladanso/Documents/GitHub/COMPLIANCEGPT/experiments/finalserverrun/final_vast_gemma3_4b.csv'
hipaa = pd.read_csv(HIPAA_PATH)

# Extract boolean features from scenario_json
BOOL_KEYS = [
    'same_org', 'is_business_associate', 'is_guardian_of_subject',
    'obtained_authorization_164_508', 'has_court_order',
    'has_lawful_process_with_assurance', 'has_lawful_process_without_assurance',
    'made_reasonable_effort_to_notify', 'is_professional_who_rendered_service',
]

def extract_bool_features(row):
    try:
        d = json.loads(row) if isinstance(row, str) else row
        return [int(bool(d.get(k, False))) for k in BOOL_KEYS]
    except:
        return [0] * len(BOOL_KEYS)

X_hipaa = np.array([extract_bool_features(sj) for sj in hipaa['scenario_json']])
y_hipaa = (hipaa['ground_truth'] == 'PERMITTED').astype(int).values

print(f'HIPAA features: {X_hipaa.shape} ({len(BOOL_KEYS)} boolean oracle predicates)')
print(f'Labels: {y_hipaa.sum()} PERMITTED, {(1-y_hipaa).sum()} DENIED')
print()

models_hipaa = {
    'Logistic Regression':        LogisticRegression(random_state=42),
    'Decision Tree (depth=5)':    DecisionTreeClassifier(max_depth=5, random_state=42),
    'MLP (16×8)':                 MLPClassifier(hidden_layer_sizes=(16, 8), max_iter=2000, random_state=42),
    'Deep MLP (64×32×16)':        MLPClassifier(hidden_layer_sizes=(64, 32, 16), max_iter=2000, random_state=42),
}

print('HIPAA compliance prediction from extracted boolean features:')
print()
scaler_h = StandardScaler()
X_hipaa_s = scaler_h.fit_transform(X_hipaa)

for name, model in models_hipaa.items():
    X_in = X_hipaa_s if 'MLP' in name else X_hipaa
    scores = cross_val_score(model, X_in, y_hipaa, cv=5, scoring='accuracy')
    bar = '█' * int(scores.mean() * 30)
    print(f'  {name:<30} {scores.mean():.1%}  {bar}')

print()
print(f'  LLM extraction → formal engine (actual system): {(hipaa["match"]=="Y").mean():.1%}')

---
## Summary

| Concept | Takeaway |
|---|---|
| **Why neural nets** | Linear models can't learn nonlinear patterns. Hidden layers can. |
| **Backpropagation** | Forward (predict) → loss → backward (gradients) → update weights. Repeat. |
| **Depth** | More layers = more complex functions. Each layer refines the representation. |
| **Scale** | Same math, 10B× more parameters, trained on 10T tokens = LLMs. |
| **Still limited** | Neural nets need numeric features. We still need text → numbers. |

> **Transition:** "We've hit the same wall again. We need to convert text to numbers in a smarter way than just counting words. Tomorrow: NLP — Bag of Words, TF-IDF, and why they fail on legal synonyms."